<a href="https://colab.research.google.com/github/weagan/Share-PEFT/blob/main/forgetting_continual_LoRA_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding
from datasets import load_dataset
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np

## 1. Standard LoRA (Stage 1)
class SimpleLoRALinear(nn.Module):
    def __init__(self, weight, rank=16):
        super().__init__()
        self.out_features, self.in_features = weight.shape
        self.register_buffer("weight", weight.clone())
        # Initialize lora_A and lora_B on the same device as the weight
        self.lora_A = nn.Parameter(torch.randn(rank, self.in_features, device=weight.device) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(self.out_features, rank, device=weight.device))
        self.scaling = 0.05

    def forward(self, x):
        delta_w = self.lora_B @ self.lora_A
        return x @ (self.weight + delta_w * self.scaling).T

## 2. Subspace LoRA (Stage 2 & 3)
class SubspaceLoRALinear(nn.Module):
    def __init__(self, weight, rank=16):
        super().__init__()
        self.out_features, self.in_features = weight.shape
        self.register_buffer("weight", weight.clone())
        self.rank = rank
        # Initialize basis and coefficients on the same device as the weight
        self.register_buffer("basis", torch.zeros(rank, self.out_features * self.in_features, device=weight.device))
        self.coefficients = nn.Parameter(torch.zeros(rank, device=weight.device))
        self.scaling = 0.05

    def forward(self, x):
        delta_w = (self.coefficients @ self.basis).view(self.out_features, self.in_features)
        return x @ (self.weight + delta_w * self.scaling).T

## 3. Manager
class ContinualSubspaceManager:
    def __init__(self, model_name="distilbert-base-uncased", rank=16):
        # Determine the primary device (cuda:0 if any GPU, else cpu)
        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

        # Load model and move to primary device
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
        self.model.to(self.device)

        # Check for multiple GPUs and wrap with DataParallel if available
        self.num_gpus = torch.cuda.device_count()
        if self.num_gpus > 1:
            print(f"Using {self.num_gpus} GPUs with DataParallel.")
            self.model = nn.DataParallel(self.model)

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.rank = rank
        self.task_coefficients = {}

        # Freeze the base model parameters.
        # If DataParallel is used, freeze the parameters of the underlying module.
        base_model_for_freezing = self.model.module if isinstance(self.model, nn.DataParallel) else self.model
        for param in base_model_for_freezing.parameters():
            param.requires_grad = False

    def _get_target_module_and_attr(self, name, model_obj):
        """Helper to get the actual parent module and attribute name to modify."""
        parts = name.split('.')
        current_obj = model_obj
        for part in parts[:-1]:
            current_obj = getattr(current_obj, part)
        return current_obj, parts[-1]

    def inject_standard_lora(self):
        # Work on the base model, whether it's wrapped by DataParallel or not
        target_model = self.model.module if isinstance(self.model, nn.DataParallel) else self.model
        for name, module in list(target_model.named_modules()): # Use list to allow modification during iteration
            if any(tgt_name in name for tgt_name in ["attention.out_lin", "attention.v_lin"]):
                parent, attr_name = self._get_target_module_and_attr(name, target_model)
                setattr(parent, attr_name, SimpleLoRALinear(module.weight, self.rank))

    def convert_to_subspace(self):
        target_model = self.model.module if isinstance(self.model, nn.DataParallel) else self.model
        with torch.no_grad():
            for name, module in list(target_model.named_modules()):
                if isinstance(module, SimpleLoRALinear):
                    delta_w = module.lora_B @ module.lora_A
                    # Perform SVD on CPU to be safer with memory, then move results back to device
                    U, S, Vh = torch.linalg.svd(delta_w.cpu(), full_matrices=False)
                    new_module = SubspaceLoRALinear(module.weight, self.rank) # Initialized on correct device
                    for r in range(self.rank):
                        # Ensure outer product is on the correct device.
                        new_module.basis[r] = torch.outer(U[:, r], Vh[r, :]).flatten().to(module.weight.device)
                    parent, attr_name = self._get_target_module_and_attr(name, target_model)
                    setattr(parent, attr_name, new_module)

    def train_task(self, task_name, loader, is_subspace=True, epochs=1):
        print(f"\n🔥 Training {task_name} ({'Subspace' if is_subspace else 'Standard LoRA'})")
        keyword = "coefficients" if is_subspace else "lora_"
        # Collect parameters from the actual trainable module
        trainable_model = self.model.module if isinstance(self.model, nn.DataParallel) else self.model
        params = [p for n, p in trainable_model.named_parameters() if keyword in n]
        optimizer = optim.AdamW(params, lr=1e-3)

        self.model.train()
        for epoch in range(epochs):
            for batch in tqdm(loader, leave=False):
                optimizer.zero_grad()
                labels = batch.pop("labels").to(self.device)
                inputs = {k: v.to(self.device) for k, v in batch.items()}
                # The model (DataParallel or not) takes inputs and labels
                outputs = self.model(**inputs, labels=labels)
                outputs.loss.backward()
                optimizer.step()

        if is_subspace:
            # Store coefficients from the actual module
            trainable_model = self.model.module if isinstance(self.model, nn.DataParallel) else self.model
            self.task_coefficients[task_name] = {n: p.clone().detach() for n, p in trainable_model.named_parameters() if "coefficients" in n}

    def evaluate_task(self, loader):
        self.model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for batch in loader:
                labels = batch.pop("labels").to(self.device)
                inputs = {k: v.to(self.device) for k, v in batch.items()}
                outputs = self.model(**inputs)
                correct += (outputs.logits.argmax(-1) == labels).sum().item()
                total += labels.size(0)
        return correct / total

    def load_task_weights(self, task_name):
        if task_name in self.task_coefficients:
            with torch.no_grad():
                # Load weights into the actual module
                target_model = self.model.module if isinstance(self.model, nn.DataParallel) else self.model
                for n, p in target_model.named_parameters():
                    if n in self.task_coefficients[task_name]:
                        p.copy_(self.task_coefficients[task_name][n])

## 4. Execution Logic
def get_loader(task, tokenizer):
    ds = load_dataset("glue", task)
    def tokenize_fn(ex):
        t = (ex["sentence"],) if "sentence" in ex else (ex["sentence1"], ex["sentence2"])
        res = tokenizer(*t, truncation=True, padding=False)
        res["labels"] = ex["label"]
        return res
    tok = ds.map(tokenize_fn, batched=True, remove_columns=ds["train"].column_names)
    return DataLoader(tok["train"], batch_size=16, shuffle=True, collate_fn=DataCollatorWithPadding(tokenizer)), \
           DataLoader(tok["validation"], batch_size=16, collate_fn=DataCollatorWithPadding(tokenizer))

manager = ContinualSubspaceManager()
tasks = ["cola", "mrpc", "sst2"]
loaders = {t: get_loader(t, manager.tokenizer) for t in tasks}
forgetting_table = np.zeros((3, 3))

# Stage 1: CoLA Warmup
manager.inject_standard_lora()
manager.train_task("cola", loaders["cola"][0], is_subspace=False)
manager.convert_to_subspace() # Creates the shared Basis

# Stage 2 & 3: Train others in subspace and evaluate
for i, task in enumerate(tasks):
    if i > 0: # CoLA is already trained
        manager.train_task(task, loaders[task][0], is_subspace=True)

    for j in range(i + 1):
        prev_task = tasks[j]
        manager.load_task_weights(prev_task)
        acc = manager.evaluate_task(loaders[prev_task][1])
        forgetting_table[i, j] = acc

## 5. Result Display
print("\n--- FORGETTING TABLE (Accuracy) ---")
print(f"{'State':<12} | {'CoLA':<6} | {'MRPC':<6} | {'SST2':<6}")
for i, row in enumerate(forgetting_table):
    row_str = " | ".join([f"{v:.3f}" if v > 0 else "-----" for v in row])
    print(f"After {tasks[i]:<7} | {row_str}")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

...

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]


🔥 Training cola (Standard LoRA)



🔥 Training mrpc (Subspace)



🔥 Training sst2 (Subspace)



--- FORGETTING TABLE (Accuracy) ---
State        | CoLA   | MRPC   | SST2  
After cola    | 0.690 | ----- | -----
After mrpc    | 0.691 | 0.684 | -----
After sst2    | 0.691 | 0.684 | 0.509
